# The BRS combustor: acoustic and intrinsic thermoacoustic modes

The BRS burner is a perfectly premixed, swirl-stabilized 30 kW test rig at TU Munich.
Emmert, Bomberg, Jaensch and Polifke used it to make a point that changed how thermoacoustic spectra are read: the eigenmodes of a combustor are **not** all acoustic.
Some are *intrinsic* (ITA) modes, sustained by a feedback loop that runs entirely through the flame: a velocity perturbation at the burner mouth makes a heat-release fluctuation, which radiates an upstream-travelling acoustic wave, which perturbs the velocity again: and that loop closes even when the boundaries reflect nothing.

This notebook rebuilds the rig in Nefes and reproduces the reference's three quantitative figures:

* **their Fig. 4**: the eigenvalues of the full system, of the pure acoustic system, and of the pure ITA system;
* **their Fig. 5**: the modulation sweep that connects each full-system mode to its acoustic or intrinsic parent;
* **their Fig. 7**: the eigenvalue tracks as the outlet reflection is dialled from a fully reflective open end to a non-reflective one, ending in the paradox that *increasing* the acoustic losses destabilizes the rig.

The inputs are the reference's Table 1 and its flame transfer function, which the paper publishes only as a figure (its Fig. 2).
The published eigenvalues we compare against are read from the same figures as printed in the first author's dissertation, where they are vector graphics: the coordinates come from the drawing commands, calibrated against the axis gridlines, so the comparison targets carry no pixel error.
They ship with Nefes as `data/brs_published_*.csv`.

> T. Emmert, S. Bomberg, S. Jaensch and W. Polifke, *Acoustic and intrinsic thermoacoustic modes of a premixed combustor*, Proc. Combust. Inst. **36** (2017) 3835–3842, [doi:10.1016/j.proci.2016.08.002](https://doi.org/10.1016/j.proci.2016.08.002).
> T. Emmert, *Intrinsic Thermoacoustic Feedback of Premixed Flames*, Dr.-Ing. dissertation, TU München (2016).

Two conventions of the reference matter throughout.
Growth rates are quoted in **hertz**, $\sigma/2\pi$ with $s = \sigma + \mathrm{i}\omega$, and positive growth means unstable.
And a mode counts as *dominant* when its growth rate exceeds the $-25\;\mathrm{Hz}$ dashed line of their Fig. 4.

## What the reference specifies

Table 1 of the paper fixes the network completely:

| Element | Parameter |
|---|---|
| Closed end | $R = +1$; inlet $M = 0.0011$, $\overline{c} = 343\;\mathrm{m/s}$, $\overline{\varrho} = 1.2\;\mathrm{kg/m^3}$ |
| Plenum | duct, $l_P = 0.17\;\mathrm{m}$ |
| Area 1 | area ratio $\alpha_1 = A_u/A_d = 29.76$ |
| Swirler tube | duct, $l_S = 0.18\;\mathrm{m}$ |
| Area 2 | area ratio $\alpha_2 = A_u/A_d = 0.13$ |
| Flame | $\theta = (\overline{T}_d - \overline{T}_u)/\overline{T}_u = 5.59$, $\xi = \overline{\varrho}_u\overline{c}_u/(\overline{\varrho}_d\overline{c}_d) = 2.57$ |
| Chamber | duct, $l_C = 0.70\;\mathrm{m}$ |
| Open end | $R = -1$ |
| FTF | *identified from LES, see their Fig. 2* |

![Network topology](brs_combustor_topology.png)

Everything but the last row is a number, and the table is over-determined in a way that confirms the reading.
For a calorically perfect gas at constant pressure, $\xi$ is not independent of $\theta$: $\xi = \sqrt{1+\theta} = 2.567$, against the quoted $2.57$.
The area ratios together with the plenum radius $r = 0.10\;\mathrm{m}$ (stated where the reference estimates the cut-on frequency) give a chamber cross-section of $90.1 \times 90.1\;\mathrm{mm}$, and the BRS chamber is $90 \times 90\;\mathrm{mm}$.
And $\overline{\varrho} = 1.2$ with $\overline{c} = 343\;\mathrm{m/s}$ puts the rig at one atmosphere.
All area changes are compact and lossless, and the model is zero-Mach; Nefes keeps its Mach-number terms, which shift the modes by less than a hertz at $M \sim 10^{-3}$.

In [ ]:
import math
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.elements import finite_impulse_response, heat_release_response
from nefes.perturbation import PerturbationBC, fit_impulse_response
from nefes.plotting import use_nefes_theme

use_nefes_theme()

DATA = Path("data")
TWO_PI = 2.0 * math.pi

# Table 1 of the reference
R_AIR, GAMMA = 287.05, 1.4
CP = GAMMA * R_AIR / (GAMMA - 1.0)
C_COLD, RHO_COLD, MACH_INLET = 343.0, 1.2, 0.0011
THETA = 5.59
XI = math.sqrt(1.0 + THETA)
L_PLENUM, L_SWIRLER, L_CHAMBER = 0.17, 0.18, 0.70
ALPHA_1, ALPHA_2 = 29.76, 0.13
R_PLENUM = 0.10

# the mean state Table 1 implies
T_COLD = C_COLD**2 / (GAMMA * R_AIR)
P_MEAN = RHO_COLD * R_AIR * T_COLD
T_HOT = (1.0 + THETA) * T_COLD
A_PLENUM = math.pi * R_PLENUM**2
A_SWIRLER = A_PLENUM / ALPHA_1
A_CHAMBER = A_SWIRLER / ALPHA_2
MDOT = RHO_COLD * MACH_INLET * C_COLD * A_PLENUM
QDOT = MDOT * CP * (T_HOT - T_COLD)

print(
    f"chamber side length {math.sqrt(A_CHAMBER)*1e3:.1f} mm, p = {P_MEAN:.0f} Pa, "
    f"mdot = {MDOT*1e3:.2f} g/s, Qdot = {QDOT/1e3:.1f} kW"
)

In [ ]:
def published(csv, kind):
    """Published eigenvalues of one marker family as ``[(frequency, growth)]`` in Hz."""
    out = []
    for line in (DATA / csv).read_text().splitlines():
        if line.startswith("#") or "," not in line or line.split(",")[1].lstrip("+-").split(".")[0].isdigit() is False:
            continue
        label, g, f = line.split(",")
        if label == kind:
            out.append((float(f), float(g)))
    return sorted(out)


PUB_FULL = published("brs_published_eigenvalues.csv", "full")
PUB_ACOUSTIC = published("brs_published_eigenvalues.csv", "acoustic")
PUB_ITA = published("brs_published_eigenvalues.csv", "ita")
len(PUB_FULL), len(PUB_ACOUSTIC), len(PUB_ITA)

## 1. The mean state

Nefes does not take Table 1's mean state as given: it solves for it.
The inputs are the mass flow (from the quoted inlet Mach number) and the heat power (from the quoted temperature ratio), and the solver returns the operating point on the network.

In [ ]:
def build_brs_rig():
    """Build the BRS network of the reference's Table 1 (no dynamic source attached yet).

    Returns the network together with the flame's node index and reference edge, since both
    are needed later to attach (or re-attach) its dynamic source in place.
    """
    net = nefes.Network(nefes.perfect_gas(R_AIR, GAMMA))
    i_inlet = net.add(cat.mass_flow_inlet(MDOT, T_COLD, perturbation_bc=PerturbationBC.hard_wall()))
    i_plenum = net.add(cat.duct(L_PLENUM, name="plenum"))
    i_contract = net.add(cat.isentropic_area_change(name="area-1"))
    i_swirler = net.add(cat.duct(L_SWIRLER, name="swirler"))
    i_expand = net.add(cat.isentropic_area_change(name="area-2"))
    i_flame = net.add(cat.heat_release_flame(QDOT, name="flame"))
    i_chamber = net.add(cat.duct(L_CHAMBER, name="chamber"))
    # reflection(-1) is the same R = -1 as open_end(), but exposes the addressable
    # outlet.perturbation_bc.magnitude / .phase knobs the reflection sweep of section 5 sweeps.
    i_outlet = net.add(cat.pressure_outlet(P_MEAN, name="outlet", perturbation_bc=PerturbationBC.reflection(-1.0)))

    net.connect(i_inlet, i_plenum, A_PLENUM)
    net.connect(i_plenum, i_contract, A_PLENUM)
    net.connect(i_contract, i_swirler, A_SWIRLER)
    e_ref = net.connect(i_swirler, i_expand, A_SWIRLER)  # the burner mouth: the FTF reference
    net.connect(i_expand, i_flame, A_CHAMBER)
    net.connect(i_flame, i_chamber, A_CHAMBER)
    net.connect(i_chamber, i_outlet, A_CHAMBER)
    return net, i_flame, e_ref


FIR_DT = 1.0e-3  # filled below; the flame's dynamic source is attached after the FTF section
FIR = None

net, i_flame, e_ref = build_brs_rig()
sol = net.solve()
assert sol.converged
# edge 1 is the cold plenum approach, edge 6 the hot chamber products
theta_solved = (sol.edge(6)["T"] - sol.edge(1)["T"]) / sol.edge(1)["T"]
mach_solved = sol.edge(1)["M"]
print(f"theta = {theta_solved:.4f} (quoted 5.59),  inlet Mach = {mach_solved:.5f} (quoted 0.0011)")

## 2. The flame transfer function, recovered from a printed figure

The FTF relates the heat-release fluctuation to the velocity fluctuation at the burner mouth, $\widehat{q} = F(f)\,\widehat{u}_u/\overline{u}_u$ with $\widehat{q} = \widehat{\dot Q}/\overline{\dot Q}$.
The reference publishes it only as gain and phase curves (its Fig. 2), so the curves were digitized from the embedded $680 \times 407$ raster: the axes calibrated from the printed ticks, the curves traced by the darkness-weighted ink centroid per column, the trace verified by overlaying it back on the figure.
The result ships as `data/brs_ftf.csv`, and one reading correction is baked into it: **the printed phase axis is labelled in reverse**.
Taken as printed ($-6\pi, -4\pi, -2\pi$ from top to bottom) the phase would *increase* with frequency, a flame that anticipates its forcing; read in reverse order the curve starts at zero phase, as a flame response must.

Digitizing gives $F$ on the real-frequency axis, but the stability eigenproblem evaluates $F$ at complex frequency, so the samples must be carried into the complex plane by a model.
The reference names the model class itself: the FTF "is a finite impulse response model ... retrieved from LES simulation and system identification".
A finite impulse response,

$$F(f) = \sum_{j=0}^{J} h_j\, e^{-2\pi\mathrm{i} f\, j\,\Delta t},$$

is a finite sum of exponentials: *entire*, with no poles anywhere: so unlike a rational fit of the same samples it continues into the whole complex plane without manufacturing spurious modes.
The coefficients come from `fit_impulse_response`, a regularized least squares on the digitized curve: $\Delta t = 1\;\mathrm{ms}$ puts the resolvable-frequency limit at $500\;\mathrm{Hz}$, exactly the edge of the digitized band, so the fit carries no frequency content the figure cannot constrain, and a second-difference penalty keeps the response smooth rather than chasing every digitization wiggle.

In [ ]:
f_dig, gain_dig, phase_dig = np.loadtxt(DATA / "brs_ftf.csv", delimiter=",", unpack=True)
F_dig = gain_dig * np.exp(1j * phase_dig)

F_FIT = fit_impulse_response(f_dig, F_dig, duration=20.0e-3, dt=FIR_DT, smoothing=1.0e-4)
FIR = F_FIT.h
F_fit = F_FIT(f_dig)

print(
    f"{F_FIT!r}  |  rms misfit {F_FIT.rms_misfit:.4f}  |  "
    f"mean lag {(F_FIT.lags * FIR).sum() / FIR.sum() * 1e3:.2f} ms"
)

### An independent record of the same response

The dissertation prints the impulse response *behind* the same FTF as a stem plot (its Fig. 4.4), and that figure is vector graphics: the 70 samples at $\Delta t = 0.25\;\mathrm{ms}$ can be read exactly from the drawing commands.
They ship as `data/brs_impulse_response.csv` and give a second, independent record of the flame response, so each digitization validates the other.

The two gain records agree to an rms of $0.006$, far inside one printed pixel, and the two phase records agree within about two pixels ($0.31\;\mathrm{rad}$) with no systematic drift.
That residual: the precision of reading a printed figure: is the error bar of everything downstream: a one-pixel systematic phase offset ($0.135\;\mathrm{rad}$) moves the intrinsic mode by about $\pm 4\;\mathrm{Hz}$, while the Mach-number terms and the network model are worth less than a hertz.

In [ ]:
h_ref = np.loadtxt(DATA / "brs_impulse_response.csv", delimiter=",", skiprows=10, usecols=3)
DT_REF = 2.5e-4
resp_ref = (h_ref * np.exp(-2j * np.pi * np.outer(f_dig, np.arange(h_ref.size)) * DT_REF)).sum(axis=1)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06, subplot_titles=("gain", "phase [rad]"))
fig.add_scatter(
    x=f_dig,
    y=gain_dig,
    name="digitized figure",
    mode="lines",
    line=dict(color="#888", width=5),
    opacity=0.5,
    row=1,
    col=1,
)
fig.add_scatter(
    x=f_dig, y=np.abs(F_fit), name="21-coefficient fit (input)", mode="lines", line=dict(color="#d62728"), row=1, col=1
)
fig.add_scatter(
    x=f_dig,
    y=np.abs(resp_ref),
    name="dissertation impulse response",
    mode="lines",
    line=dict(color="#1f77b4", dash="dot"),
    row=1,
    col=1,
)
fig.add_scatter(
    x=f_dig, y=phase_dig, showlegend=False, mode="lines", line=dict(color="#888", width=5), opacity=0.5, row=2, col=1
)
fig.add_scatter(
    x=f_dig, y=np.unwrap(np.angle(F_fit)), showlegend=False, mode="lines", line=dict(color="#d62728"), row=2, col=1
)
fig.add_scatter(
    x=f_dig,
    y=np.unwrap(np.angle(resp_ref)),
    showlegend=False,
    mode="lines",
    line=dict(color="#1f77b4", dash="dot"),
    row=2,
    col=1,
)
fig.update_xaxes(title_text="frequency [Hz]", row=2, col=1)
fig.update_layout(height=520, margin=dict(t=40, b=40), legend=dict(orientation="h", y=1.12))
fig.show()

gain_rms = float(np.sqrt(((np.abs(resp_ref) - gain_dig) ** 2).mean()))
dphi = np.unwrap(np.angle(resp_ref)) - phase_dig
print(f"two-source agreement: gain rms {gain_rms:.4f}, max phase difference {np.abs(dphi).max():.3f} rad")

In [ ]:
fig = go.Figure()
fig.add_bar(
    x=np.arange(h_ref.size) * DT_REF * 1e3,
    y=h_ref / DT_REF,
    name="dissertation stems (exact)",
    marker_color="#1f77b4",
    opacity=0.6,
)
fig.add_scatter(
    x=F_FIT.lags * 1e3,
    y=FIR / FIR_DT,
    name="fit from the digitized FTF",
    mode="lines+markers",
    line=dict(color="#d62728"),
)
fig.update_layout(
    height=380,
    xaxis_title="time [ms]",
    yaxis_title="impulse response h(t) [1/s]",
    margin=dict(t=30, b=40),
    legend=dict(orientation="h", y=1.1),
)
fig.show()

Both records show the double lobe Komarek and Polifke describe for this burner: the positive axial-velocity response peaking near $5\;\mathrm{ms}$, then the negative swirl lobe near $8\;\mathrm{ms}$: and the peak sits at the $4.8\;\mathrm{ms}$ characteristic delay the dissertation itself quotes.
Neither feature was fitted for.

Everything below runs on the 21-coefficient fit, i.e. on the paper's own Fig. 2; the dissertation's impulse response reappears alongside it where it localizes the residual: as a second marker set in the Fig. 4 overlay and as endpoint checks in the sweeps.

## 3. Their Fig. 4: the full, the pure acoustic, and the pure ITA spectrum

The reference compares three systems.

* The **full system**: the network as built.
* The **pure acoustic system**: the flame's unsteady heat release removed ($F = 0$), the temperature jump kept.
  In Nefes that is simply the same network without the dynamic source: the flame element still adds its steady heat.
* The **pure ITA system**: the reflections removed instead.
  The burner mouth and flame are placed between anechoic ends, and whatever eigenvalues remain owe nothing to the geometry: they are the flame's own feedback loop.

All three are eigenvalue problems on a network operator, solved on the same complex-frequency window as the printed figure: frequency to $760\;\mathrm{Hz}$, growth to $-100\;\mathrm{Hz}$.

In [ ]:
def scan(sol, f_lo=5.0, f_hi=760.0, g_band=(-660.0, 80.0), chunks=9, overlap=12.0):
    """Eigenmodes over a wide window, searched in overlapping frequency chunks."""
    edges = np.linspace(f_lo, f_hi, chunks + 1)
    out = []
    for a, b in zip(edges[:-1], edges[1:]):
        res = sol.eigenmodes(
            freq_band=(max(f_lo, a - overlap), min(f_hi, b + overlap)), growth_band=g_band, isentropic=True
        )
        out += [(float(f), float(g) / TWO_PI) for f, g in zip(res.freqs, res.growth_rates)]
    dedup = []
    for f, g in sorted(out):
        if all(abs(f - f2) > 0.5 or abs(g - g2) > 0.5 for f2, g2 in dedup):
            dedup.append((f, g))
    return dedup


sol_passive = net.solve()
assert sol_passive.converged
modes_acoustic = scan(sol_passive, g_band=(-80.0, 80.0))

# Attach the flame's dynamic source once; from here on `net` is the full (active) system.
net.set_dynamic_source(i_flame, heat_release_response(finite_impulse_response(FIR, FIR_DT), ref_edge=e_ref))
sol_full = net.solve()
assert sol_full.converged
modes_full = scan(sol_full)

for f, g in modes_full:
    print(f"full: {f:7.2f} Hz  {g:+7.2f} Hz")
for f, g in modes_acoustic:
    print(f"acoustic: {f:7.2f} Hz  {g:+7.2f} Hz")

In [ ]:
def ita_rig():
    """The pure ITA system: the burner mouth and flame between anechoic ends."""
    net = nefes.Network(nefes.perfect_gas(R_AIR, GAMMA))
    i_inlet = net.add(cat.mass_flow_inlet(MDOT, T_COLD, perturbation_bc=PerturbationBC.anechoic()))
    i_expand = net.add(cat.isentropic_area_change(name="area-2"))
    i_flame = net.add(cat.heat_release_flame(QDOT))
    i_outlet = net.add(cat.pressure_outlet(P_MEAN, perturbation_bc=PerturbationBC.anechoic()))
    e_ref = net.connect(i_inlet, i_expand, A_SWIRLER)
    net.connect(i_expand, i_flame, A_CHAMBER)
    net.connect(i_flame, i_outlet, A_CHAMBER)
    net.set_dynamic_source(i_flame, heat_release_response(finite_impulse_response(FIR, FIR_DT), ref_edge=e_ref))
    return net


sol_ita = ita_rig().solve()
assert sol_ita.converged
modes_ita = scan(sol_ita)
for f, g in modes_ita:
    print(f"pure ITA: {f:7.2f} Hz  {g:+7.2f} Hz")

In [ ]:
# the same full system, driven by the reference's own impulse response instead of the
# Fig. 2 fit: this isolates what the figure digitization costs.  Swapping the impulse-response
# array is a structural change to the dynamic source (not a scalar knob), so it goes on a copy.
net_ref = net.copy()
net_ref.set_dynamic_source(i_flame, heat_release_response(finite_impulse_response(h_ref, DT_REF), ref_edge=e_ref))
sol_ref = net_ref.solve()
assert sol_ref.converged
modes_ref = scan(sol_ref)

In [ ]:
fig = go.Figure()
# the reference's dominance box: growth above -25 Hz, frequency below 500 Hz
fig.add_shape(type="rect", x0=-25, x1=25, y0=0, y1=500, line=dict(color="#999", dash="dash"))

for pts, name, color, symbol in [
    (PUB_FULL, "published full", "#444", "x-open"),
    (PUB_ACOUSTIC, "published acoustic", "#1f77b4", "circle-open"),
    (PUB_ITA, "published ITA", "#d62728", "square-open"),
]:
    fig.add_scatter(
        x=[g for _, g in pts],
        y=[f for f, _ in pts],
        mode="markers",
        name=name,
        marker=dict(symbol=symbol, size=13, color=color, line=dict(width=2)),
    )
for pts, name, color, symbol in [
    (modes_full, "Nefes full", "#444", "x"),
    (modes_acoustic, "Nefes acoustic", "#1f77b4", "circle"),
    (modes_ita, "Nefes pure ITA", "#d62728", "square"),
    (modes_ref, "Nefes full, reference impulse response", "#2ca02c", "diamond"),
]:
    fig.add_scatter(
        x=[g for _, g in pts],
        y=[f for f, _ in pts],
        mode="markers",
        name=name,
        marker=dict(symbol=symbol, size=7, color=color),
    )
fig.update_layout(
    height=520,
    xaxis_title="growth rate [Hz]",
    yaxis_title="frequency [Hz]",
    xaxis_range=[-105, 25],
    yaxis_range=[-10, 760],
    margin=dict(t=30, b=40),
)
fig.show()

Reading the spectrum inside the dominance box first, published values in parentheses:

| Mode | Nefes, Fig. 2 input | Nefes, reference impulse response | Published |
|---|---|---|---|
| Helmholtz mode of plenum and swirler tube | $43.2\;\mathrm{Hz}$ at $-9.9$ | $42.5$ at $-7.9$ | $42.4$ at $-8.2$ |
| **Intrinsic mode** | $105.7\;\mathrm{Hz}$ at $-0.8$ | $110.8$ at $-2.1$ | $111.0$ at $-2.0$ |
| Chamber quarter wave | $319.5\;\mathrm{Hz}$ at $-16.6$ | $322.6$ at $-14.9$ | $315.6$ at $-6.3$ |

The pure acoustic system has exactly two modes in the window, at $54.4$ and $320.4\;\mathrm{Hz}$ against the published $54.5$ and $320.5$, both neutral to within a few hertz as ideal lossless modes must be.
The pure ITA system contributes its own family: the chain of damped modes marching up the left of the figure: with the dominant member at $103.2\;\mathrm{Hz}$, $-21.2\;\mathrm{Hz}$ against the published $105.2$, $-22.1$.
The count works out exactly as the reference argues: the full system's modes are the union of two acoustic parents and the ITA family.

### Where the overlay does and does not match, and why

The visible offsets fall into three classes, each with its own cause, and the green diamonds: the same network driven by the reference's own impulse response instead of the Fig. 2 fit: separate them.

**Near the axis (growth above about $-30\;\mathrm{Hz}$), the residual is the figure-reading of the FTF and nothing else.**
With the reference's impulse response the Helmholtz mode lands at $(-7.9, 42.5)$ against the published $(-8.2, 42.4)$ and the intrinsic mode at $(-2.1, 110.8)$ against $(-2.0, 111.0)$: agreement to $0.3\;\mathrm{Hz}$ on both axes.
The $5\;\mathrm{Hz}$ the intrinsic family sits low with the Fig. 2 input is the $\pm$two-pixel phase reading of a printed figure, exactly the error bar estimated in section 2.

**Near $315\;\mathrm{Hz}$, the pair splitting is not a meaningful comparison.**
The quarter wave and the second ITA mode form an avoided crossing; its *mean* is robust ($310.3$ vs the published $314.9\;\mathrm{Hz}$, $-19.4$ vs $-16.5\;\mathrm{Hz}$) but the splitting between the partners is hypersensitive to the flame response: the reference's impulse response splits it $(-20.3, 301.5)$ / $(-14.9, 322.6)$ where the published figure shows $(-26.6, 314.3)$ / $(-6.3, 315.6)$.
Compare the pair through its mean, never through the partners.

**The deep-damped chain (growth below about $-45\;\mathrm{Hz}$) is not decidable from the published material.**
Those eigenvalues read the flame response at $e^{|\sigma|\tau}$ amplifications of $e^{4}$ and beyond, and they drift $10$–$25\;\mathrm{Hz}$ from the published squares *even with the reference's own printed impulse response*: the discrepancy is insensitive to the figure-reading precision and to any invisible tail, so it reflects the reference's actual model realization (its identified state-space flame model and its third-order-upwind duct discretization, whose own convergence figure in the dissertation shows exactly this deep-eigenvalue drift; the Nefes ducts are exact delay operators).
Two independent implementations on our side: the network operator's contour solver and the scalar dispersion polynomial, residual-verified: agree there to better than $0.3\;\mathrm{Hz}$, so the drift is not ours to close.

### The dispersion relation of the intrinsic mode, and a normalization the paper glosses over

For the pure ITA system the reference derives a dispersion relation that involves no geometry at all (its Eq. 9):

$$\alpha_2 + \xi + F(s)\,\theta = 0 .$$

Solving it as printed, with the published FTF, gives a dominant root with **positive** growth near $+38\;\mathrm{Hz}$: but the reference's own Fig. 4 shows every ITA mode stable.
The resolution is a normalization the paper's scalar equations gloss over.
Its jump relation feeds the heat release into the velocity balance as $\theta\,\widehat{q}$ with $\widehat{q}$ normalized by the velocity at the flame's *own* (chamber) side; the published FTF is normalized by the *burner-mouth* velocity, and the two mean velocities differ by the area ratio.
With the bridge in place the relation reads

$$\alpha_2 + \xi + \theta\,\alpha_2\,F(s) = 0,$$

and its dominant root lands on the published square.
Nefes never sees this issue: it assembles the jump conditions from the conservation laws, so the consistent coupling is automatic: the network eigenvalue above and the corrected scalar root below agree.

In [ ]:
def pure_ita_root(theta_eff, seed=(-20.0, 100.0)):
    """Newton root of ``alpha_2 + xi + theta_eff F(s) = 0`` -> (frequency, growth) in Hz."""
    lags = np.arange(FIR.size) * FIR_DT
    relation = lambda s: ALPHA_2 + XI + theta_eff * np.sum(FIR * np.exp(-s * lags))
    s = TWO_PI * (seed[0] + 1j * seed[1])
    for _ in range(60):
        h = 1e-4 * (1.0 + abs(s))
        d = (relation(s + h) - relation(s - h)) / (2.0 * h)
        step = relation(s) / d
        s -= step
        if abs(step) < 1e-10 * (1.0 + abs(s)):
            break
    return s.imag / TWO_PI, s.real / TWO_PI


f_wrong, g_wrong = pure_ita_root(THETA, seed=(30.0, 110.0))
f_fixed, g_fixed = pure_ita_root(THETA * ALPHA_2)
print(f"Eq. (9) as printed:        {f_wrong:6.1f} Hz at {g_wrong:+6.1f} Hz  (unstable: contradicts their Fig. 4)")
print(f"with the alpha_2 bridge:   {f_fixed:6.1f} Hz at {g_fixed:+6.1f} Hz  (published square: 105.2 at -22.1)")

## 4. Their Fig. 5: which full-system mode is which

Proximity in Fig. 4 does not say which full-system mode descends from which parent.
The reference settles it by rearranging the full system into coupled acoustic and ITA blocks and modulating the coupling with a parameter $\mu$ (its Eq. 18): at $\mu = 0$ the characteristic equation factorizes exactly into the pure acoustic and pure ITA relations, at $\mu = 1$ the full system returns, and each eigenvalue traces a continuous trajectory in between.
It stresses that simply scaling the flame response, $F \to \mu F$, would *not* do this: that recovers the acoustic system at $\mu = 0$ but not the ITA one.

The rearranged system is a $6\times 6$ matrix in the wave amplitudes and the flame variables, fully specified in the paper, with the up- and downstream sections entering as impedances $Z_u(s)$, $Z_d(s)$ at the burner mouth and flame exit.
Both are three-line functions of the Table-1 geometry for a zero-Mach ideal gas, so the whole construction fits in a code cell: with the same $\alpha_2$ normalization bridge on $F$ as above.

In [ ]:
TAU_P, TAU_S, TAU_C = L_PLENUM / C_COLD, L_SWIRLER / C_COLD, L_CHAMBER / (C_COLD * XI)
LAGS = np.arange(FIR.size) * FIR_DT
F_of = lambda s: np.sum(FIR * np.exp(-s * LAGS))


def duct_tm(tau, s):
    """Zero-Mach duct transfer matrix in (p/(rho c), u), left to right."""
    ch, sh = np.cosh(tau * s), np.sinh(tau * s)
    return np.array([[ch, -sh], [-sh, ch]])


def Z_up(s):
    """Impedance of the upstream section at the burner mouth (closed end side)."""
    state = duct_tm(TAU_P, s) @ np.array([1.0 + 0j, 0.0 + 0j])  # closed end: u = 0
    state[1] *= ALPHA_1  # compact contraction into the swirler tube
    state = duct_tm(TAU_S, s) @ state
    return state[0] / state[1]


def Z_down(s, r_out=-1.0):
    """Impedance of the downstream section at the flame (chamber + outlet reflection)."""
    state = duct_tm(-TAU_C, s) @ np.array([1.0 + r_out, 1.0 - r_out], dtype=complex)
    return state[0] / state[1]


def mu_det(s, mu, r_out=-1.0):
    """Determinant of the reference's Eq. (18) modulation matrix."""
    zu, zd = Z_up(s), Z_down(s, r_out)
    a, x, th = ALPHA_2, XI, THETA
    ax = a + x
    Ru = (zu + 1.0) / (zu - 1.0)
    Rd = (zd - 1.0) / (1.0 + zd)
    Ff = a * F_of(s)  # the alpha_2 bridge: Fig. 2's F is burner-mouth normalized
    A = np.array(
        [
            [-1.0, Ru, 0.0, 0.0, 0.0, mu * Ru * th / ax],
            [(a - x) / ax, -1.0, 0.0, 2.0 / ax, 0.0, 0.0],
            [2.0 * a * x / ax, 0.0, -1.0, (x - a) / ax, 0.0, mu * x * th / ax],
            [0.0, 0.0, Rd, -1.0, 0.0, 0.0],
            [mu, -mu, 0.0, 0.0, -1.0, -th / ax],
            [0.0, 0.0, 0.0, 0.0, Ff, -1.0],
        ],
        dtype=complex,
    )
    return np.linalg.det(A)


def polish(fun, s0, tol=1e-10):
    """Complex Newton with a numerical derivative."""
    s = complex(s0)
    for _ in range(80):
        h = 1e-4 * (1.0 + abs(s))
        d = (fun(s + h) - fun(s - h)) / (2.0 * h)
        step = fun(s) / d
        s -= step
        if abs(step) < tol * (1.0 + abs(s)):
            break
    return s


# the mu = 0 seeds: the pure acoustic roots and the dominant pure ITA root
seeds = {
    "Helmholtz": polish(lambda s: Z_down(s) * ALPHA_2 - XI * Z_up(s), TWO_PI * 1j * 54.0),
    "intrinsic": polish(lambda s: ALPHA_2 + XI + THETA * ALPHA_2 * F_of(s), TWO_PI * (-20 + 1j * 100.0)),
    "quarter wave": polish(lambda s: Z_down(s) * ALPHA_2 - XI * Z_up(s), TWO_PI * 1j * 320.0),
}
for name, s in seeds.items():
    print(f"mu = 0 seed {name}: {s.imag/TWO_PI:7.2f} Hz at {s.real/TWO_PI:+7.2f} Hz")

In [ ]:
MU = np.linspace(0.0, 1.0, 51)
trajectories = {}
for name, s in seeds.items():
    pts = []
    for mu in MU:
        s = polish(lambda x: mu_det(x, mu), s)
        pts.append((s.real / TWO_PI, s.imag / TWO_PI))
    trajectories[name] = np.array(pts)

# at mu = 1 the trajectories must land on the full-system eigenvalues of the network;
# the residual is the zero-Mach idealization of the scalar model, largest for the
# quarter wave, whose avoided-crossing partner sits a few hertz away
for name, pts in trajectories.items():
    g1, f1 = pts[-1]
    f_net, g_net = min(modes_full, key=lambda m: abs(m[0] - f1))
    assert abs(f1 - f_net) < 2.5 and abs(g1 - g_net) < 6.0, (name, pts[-1], (f_net, g_net))
    print(f"{name}: mu=1 endpoint ({g1:+6.2f}, {f1:6.2f}) Hz vs network ({g_net:+6.2f}, {f_net:6.2f}) Hz")

In [ ]:
pub_sweep = published("brs_published_mu_sweep.csv", "sweep")
pub_mu0_ac = published("brs_published_mu_sweep.csv", "acoustic-at-mu-0")
pub_mu0_ita = published("brs_published_mu_sweep.csv", "ita-at-mu-0")
pub_mu1 = published("brs_published_mu_sweep.csv", "full-at-mu-1")

fig = go.Figure()
fig.add_scatter(
    x=[g for _, g in pub_sweep],
    y=[f for f, _ in pub_sweep],
    mode="markers",
    name="published sweep",
    marker=dict(size=4, color="#aaa"),
)
fig.add_scatter(
    x=[g for _, g in pub_mu0_ac],
    y=[f for f, _ in pub_mu0_ac],
    mode="markers",
    name="published acoustic (mu = 0)",
    marker=dict(symbol="circle-open", size=13, color="#1f77b4", line=dict(width=2)),
)
fig.add_scatter(
    x=[g for _, g in pub_mu0_ita],
    y=[f for f, _ in pub_mu0_ita],
    mode="markers",
    name="published ITA (mu = 0)",
    marker=dict(symbol="square-open", size=13, color="#d62728", line=dict(width=2)),
)
fig.add_scatter(
    x=[g for _, g in pub_mu1],
    y=[f for f, _ in pub_mu1],
    mode="markers",
    name="published full (mu = 1)",
    marker=dict(symbol="x-open", size=13, color="#444", line=dict(width=2)),
)
colors = {"Helmholtz": "#1f77b4", "intrinsic": "#d62728", "quarter wave": "#2ca02c"}
for name, pts in trajectories.items():
    fig.add_scatter(
        x=pts[:, 0],
        y=pts[:, 1],
        mode="lines+markers",
        name=f"Nefes: {name}",
        marker=dict(size=4, color=colors[name]),
        line=dict(color=colors[name], width=1),
    )
fig.update_layout(
    height=500,
    xaxis_title="growth rate [Hz]",
    yaxis_title="frequency [Hz]",
    xaxis_range=[-25, 25],
    yaxis_range=[0, 500],
    margin=dict(t=30, b=40),
)
fig.show()

The three trajectories reproduce the published sweep: the two acoustic modes are *stabilized* by coupling to the flame (their growth rates walk from neutral to $-9$ and $-12\;\mathrm{Hz}$), while the intrinsic mode is *destabilized* (from $-22$ up to $-0.8\;\mathrm{Hz}$), and no trajectory jumps branches.
That establishes the one-to-one correspondence behind the labels used throughout: the $43$ and $319\;\mathrm{Hz}$ modes of the full rig are acoustic in origin, the $106\;\mathrm{Hz}$ mode: the least damped mode of the rig, the one the experiment saw oscillate near $100\;\mathrm{Hz}$: is intrinsic.
The trajectory endpoints and the published markers differ by the same few hertz as in Fig. 4, inherited from the digitized flame response; the trajectory *shapes* match throughout.

## 5. Their Fig. 7: the paradox of the helpful losses

The last figure of the reference sweeps the outlet reflection from a fully reflective ideal open end, $R = -1$, toward a non-reflective outlet, $R = 0$: as if the rig exhaust were progressively lined with a perfect absorber.
Every acoustic argument says this should stabilize the system: energy that leaves through the outlet is lost to the resonance.

Nefes marches the eigenvalues along the sweep by continuation: each mode is followed with a predictor-corrector step from the previous reflection value, which keeps the branches identified through the crossings.

In [ ]:
R_MAG = np.linspace(1.0, 0.0, 51)
# The outlet's reflection was built as PerturbationBC.reflection(-1.0), so its magnitude is
# already the addressable knob this sweep needs; the phase (pi, i.e. R < 0) stays fixed.
traj = net.eigenvalue_trajectory(
    "outlet.perturbation_bc.magnitude",
    R_MAG,
    freq_band=(5.0, 495.0),
    growth_band=(-340.0, 170.0),
    isentropic=True,
    param_name="|R| at the outlet",
)
for br in traj.branches:
    f0, g0 = br.freqs[0], br.growth[0] / TWO_PI
    f1, g1 = br.freqs[-1], br.growth[-1] / TWO_PI
    print(f"branch from ({g0:+6.2f}, {f0:6.2f}) Hz to ({g1:+6.2f}, {f1:6.2f}) Hz  alive={br.alive}")

In [ ]:
pub_dots = published("brs_published_reflection_sweep.csv", "sweep")
pub_r1_ac = published("brs_published_reflection_sweep.csv", "acoustic-at-full-reflection")
pub_r1_ita = published("brs_published_reflection_sweep.csv", "ita-at-full-reflection")
pub_r0 = published("brs_published_reflection_sweep.csv", "at-zero-reflection")

fig = go.Figure()
fig.add_shape(type="line", x0=-25, x1=-25, y0=0, y1=500, line=dict(color="#999", dash="dash"))
fig.add_scatter(
    x=[g for _, g in pub_dots],
    y=[f for f, _ in pub_dots],
    mode="markers",
    name="published sweep",
    marker=dict(size=4, color="#aaa"),
)
fig.add_scatter(
    x=[g for _, g in pub_r1_ac],
    y=[f for f, _ in pub_r1_ac],
    mode="markers",
    name="published acoustic (R = -1)",
    marker=dict(symbol="circle-open", size=13, color="#1f77b4", line=dict(width=2)),
)
fig.add_scatter(
    x=[g for _, g in pub_r1_ita],
    y=[f for f, _ in pub_r1_ita],
    mode="markers",
    name="published ITA (R = -1)",
    marker=dict(symbol="square-open", size=13, color="#d62728", line=dict(width=2)),
)
fig.add_scatter(
    x=[g for _, g in pub_r0],
    y=[f for f, _ in pub_r0],
    mode="markers",
    name="published non-reflective end",
    marker=dict(symbol="x-open", size=13, color="#444", line=dict(width=2)),
)
for br in traj.branches:
    fig.add_scatter(
        x=np.asarray(br.growth) / TWO_PI,
        y=br.freqs,
        mode="lines+markers",
        name=f"Nefes branch {br.freqs[0]:.0f} Hz",
        marker=dict(size=4),
        line=dict(width=1),
    )
fig.update_layout(
    height=520,
    xaxis_title="growth rate [Hz]",
    yaxis_title="frequency [Hz]",
    xaxis_range=[-52, 27],
    yaxis_range=[-5, 500],
    margin=dict(t=30, b=40),
)
fig.show()

In [ ]:
# the headline number: the intrinsic mode at the non-reflective outlet
br_ita = min(traj.branches, key=lambda b: abs(b.freqs[0] - 105.7))
f_end, g_end = br_ita.freqs[-1], br_ita.growth[-1] / TWO_PI
f_pub, g_pub = min(
    published("brs_published_reflection_sweep.csv", "at-zero-reflection"), key=lambda m: abs(m[0] - f_end)
)
neutral = R_MAG[np.argmin(np.abs(np.asarray(br_ita.growth)))]
print(f"intrinsic mode at R = 0:  Nefes ({g_end:+5.2f}, {f_end:5.1f}) Hz,  published ({g_pub:+5.2f}, {f_pub:5.1f}) Hz")
print(f"neutral crossing at |R| = {neutral:.2f}")

The sweep reproduces the published tracks, and with them the paradox.
Both acoustic branches move *left*: the Helmholtz mode dives to a strongly damped low-frequency mode, and the quarter wave damps past $-40\;\mathrm{Hz}$.
The one visible difference sits at the bottom of the figure: the published Helmholtz track reaches the frequency axis and splits into two non-oscillating modes, while ours stops just short of it, at $33\;\mathrm{Hz}$ and the same damping level.
At those growth rates the eigenvalue reads the flame's impulse response amplified by $e^{|\sigma|\tau} \sim e^{5}$, far beyond what either digitized record of the FTF can constrain, so whether the collision with the axis completes is not decidable from the printed material.
The intrinsic branch moves *right*: it crosses into instability near $|R| = 0.92$ and ends, at the perfectly absorbing outlet, unstable at $(+23.7, 83.5)\;\mathrm{Hz}$ against the published $(+23.0, 88.5)$: agreement to better than a hertz in growth rate, with the familiar few hertz of FTF digitization in frequency.
Driving the same sweep endpoint with the reference's own impulse response gives $(+22.6, 88.5)$: the frequency offset closes, as in Fig. 4.

Note what the endpoint is and is not.
With the outlet absorbing, the intrinsic loop is still closed through the *upstream* reflections: this is not the pure ITA system of Fig. 4, whose two-sided anechoic mode is stable at $-21\;\mathrm{Hz}$: and that one-sided configuration happens to be unstable.
So the paradox is a statement about coupling, not about losses: the downstream acoustics were bleeding energy out of the intrinsic feedback, and opening the boundary removes the drain.
Acoustic damping treatments that work by making boundaries less reflective can therefore destabilize a combustor, which is the reference's closing warning.

## Summary

| Quantity | Nefes | Published | Where the residual lives |
|---|---|---|---|
| Temperature ratio $\theta$ | $5.5898$ | $5.59$ |: |
| Inlet Mach | $0.00110$ | $0.0011$ |: |
| Pure acoustic modes | $54.4$, $320.4\;\mathrm{Hz}$ | $54.5$, $320.5\;\mathrm{Hz}$ | ideal-mode growth within $3\;\mathrm{Hz}$ of neutral in both codes |
| Full: Helmholtz | $43.2\;\mathrm{Hz}$ at $-9.9$ | $42.4$ at $-8.2$ | FTF digitization |
| Full: intrinsic | $105.7\;\mathrm{Hz}$ at $-0.8$ | $111.0$ at $-2.0$ | FTF phase pixels ($110.8$ at $-2.1$ with the dissertation's impulse response) |
| Full: quarter-wave pair mean | $310.3\;\mathrm{Hz}$ at $-19.4$ | $314.9$ at $-16.5$ | avoided-crossing splitting is FTF-hypersensitive |
| Dominant pure ITA mode | $103.2\;\mathrm{Hz}$ at $-21.2$ | $105.2$ at $-22.1$ | FTF digitization |
| $\mu$-sweep trajectories | acoustic stabilized, ITA destabilized, no branch jumps | same | endpoint offsets as above |
| Intrinsic mode at $R = 0$ | $83.5\;\mathrm{Hz}$ at $+23.7$ | $88.5$ at $+23.0$ | FTF digitization ($88.5$ at $+22.6$ with the dissertation's impulse response) |
| Neutral crossing of the ITA branch | $\lvert R\rvert \approx 0.92$ | between the published dots at $\lvert R\rvert \approx 0.9$ |: |

Beyond the numbers, reproducing the reference surfaced two quirks of the printed paper worth recording.
The phase axis of its FTF figure is labelled in reverse (the curve read as printed would anticipate its forcing), and its scalar dispersion relations omit the normalization bridge $\alpha_2$ between the published burner-mouth FTF and the flame-side heat-release variable: Eq. (9) as printed, fed with the paper's own FTF, predicts an unstable pure ITA mode in contradiction with the paper's own figure.
Neither affects the reference's conclusions; both matter when reproducing it from the printed material alone.

## References

* T. Emmert, S. Bomberg, S. Jaensch, W. Polifke, *Acoustic and intrinsic thermoacoustic modes of a premixed combustor*, Proc. Combust. Inst. 36 (2017) 3835–3842.
* T. Emmert, *Intrinsic Thermoacoustic Feedback of Premixed Flames*, Dr.-Ing. dissertation, TU München, 2016.
* S. Bomberg, T. Emmert, W. Polifke, *Thermal versus acoustic response of velocity sensitive premixed flames*, Proc. Combust. Inst. 35 (2015) 3185–3192.
* T. Komarek, W. Polifke, *Impact of swirl fluctuations on the flame response of a perfectly premixed swirl burner*, J. Eng. Gas Turbines Power 132 (2010) 061503.
* L. Tay-Wo-Chong, S. Bomberg, A. Ulhaq, T. Komarek, W. Polifke, *Comparative validation study on identification of premixed flame transfer function*, J. Eng. Gas Turbines Power 134 (2012) 021502.